In [16]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd()
IN_DIR = ROOT / "Data" / "extracted"
OUT_DIR = ROOT / "Data" / "processed"

OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Reading .xlsx from:", IN_DIR)
print("Saving outputs to:", OUT_DIR)


Reading .xlsx from: c:\Users\sabar\Desktop\Documents\Personal\Electricity demand\Data\extracted
Saving outputs to: c:\Users\sabar\Desktop\Documents\Personal\Electricity demand\Data\processed


In [17]:
PERIMETER_COL = "PERIMETER"      # may exist; if not, we'll use filename as perimeter
DATE_COL = "DATE"
TIME_COL = "HOUR"
CONS_COL = "CONSUMPTION"

In [50]:
def build_datetime_from_date_hour(date_series, hour_series):
    # DATE is datetime64 -> normalize to midnight
    date = pd.to_datetime(date_series, errors="coerce").dt.normalize()

    # HOUR is string like "16:30:00" -> timedelta
    hour = pd.to_timedelta(hour_series.astype(str).str.strip(), errors="coerce")

    # Handle 24:00 if it exists (rare, but safe)
    mask_24 = hour_series.astype(str).str.strip().eq("24:00") | hour_series.astype(str).str.strip().eq("24:00:00")
    hour = hour.where(~mask_24, pd.to_timedelta("00:00:00"))
    
    dt = date + hour
    dt.loc[mask_24] = dt.loc[mask_24] + pd.Timedelta(days=1)

    return dt


In [51]:
def slugify(name: str) -> str:
    name = name.strip().lower().replace(" ", "_").replace("-", "_")
    return "".join(ch for ch in name if ch.isalnum() or ch == "_")

def to_hourly_mwh(df: pd.DataFrame, perimeter_name: str) -> pd.DataFrame:
    """
    Converts irregular 15-min / 30-min power readings (MW) into hourly energy (MWh)
    using time-delta integration. Robust to missing 15-min rows.
    """
    df = df.copy()

    # If perimeter column exists, keep it; otherwise use filename perimeter
    if PERIMETER_COL in df.columns:
        df["Perimeter"] = df[PERIMETER_COL].astype(str).str.strip()
    else:
        df["Perimeter"] = perimeter_name

    # Keep only needed
    needed = [DATE_COL, TIME_COL, CONS_COL, "Perimeter"]
    missing = [c for c in [DATE_COL, TIME_COL, CONS_COL] if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns {missing}. Found: {list(df.columns)}")

    df = df[needed].copy()

    # Build datetime (French date: dd-mm-yyyy) + HH:MM
    df["dt"] = build_datetime_from_date_hour(df[DATE_COL], df[TIME_COL])


    df = df.dropna(subset=["dt"])

    # Parse consumption numeric
    df[CONS_COL] = pd.to_numeric(df[CONS_COL], errors="coerce")
    df = df.dropna(subset=[CONS_COL])

    # Sort within each perimeter
    df = df.sort_values(["Perimeter", "dt"]).reset_index(drop=True)

    # Compute delta time to next point (hours), per perimeter
    next_dt = df.groupby("Perimeter")["dt"].shift(-1)
    delta_h = (next_dt - df["dt"]).dt.total_seconds() / 3600.0

    # Cap deltas to avoid crazy gaps (optional safety):
    # If there's a gap > 1 hour, we don't want to allocate huge energy to one measurement.
    # We'll set those to NaN and drop them.
    delta_h = delta_h.where(delta_h <= 1.0)

    df["delta_h"] = delta_h

    # Energy for that interval (MWh) = MW * hours
    df["energy_mwh"] = df[CONS_COL] * df["delta_h"]

    # Drop last row per perimeter (no next_dt => NaN) + any weird gaps
    df = df.dropna(subset=["energy_mwh"])

    # Allocate energy to the hour bucket of the start timestamp
    df["Date"] = df["dt"].dt.date.astype(str)
    df["Hour"] = df["dt"].dt.hour.astype(int)

    hourly = (
        df.groupby(["Date", "Hour", "Perimeter"], as_index=False)["energy_mwh"]
          .sum()
          .rename(columns={"energy_mwh": "Consumption_MWh"})
          .sort_values(["Perimeter", "Date", "Hour"])
          .reset_index(drop=True)
    )

    return hourly[["Date", "Hour", "Perimeter", "Consumption_MWh"]]


In [52]:
xlsx_files = sorted(IN_DIR.glob("*.xlsx"))
print("Found xlsx files:", len(xlsx_files))

for f in xlsx_files:
    perimeter_name = f.stem  # filename without extension
    print("Processing:", f.name)

    df_raw = pd.read_excel(f, engine="openpyxl")

    hourly = to_hourly_mwh(df_raw, perimeter_name=perimeter_name)

    out_path = OUT_DIR / f"{slugify(perimeter_name)}_hourly.csv"
    hourly.to_csv(out_path, index=False)

    print("✅ Saved:", out_path, "| rows:", len(hourly))

Found xlsx files: 13
Processing: Auvergne Rhone Alpes.xlsx
✅ Saved: Data\processed\auvergne_rhone_alpes_hourly.csv | rows: 26304
Processing: Bourgogne-Franche-Comté.xlsx
✅ Saved: Data\processed\bourgogne_franche_comté_hourly.csv | rows: 26304
Processing: Bretagne.xlsx
✅ Saved: Data\processed\bretagne_hourly.csv | rows: 26304
Processing: Centre-Val-de-Loire.xlsx
✅ Saved: Data\processed\centre_val_de_loire_hourly.csv | rows: 26304
Processing: Grand-Est.xlsx
✅ Saved: Data\processed\grand_est_hourly.csv | rows: 26304
Processing: Hauts-de-France.xlsx
✅ Saved: Data\processed\hauts_de_france_hourly.csv | rows: 26304
Processing: Ile-de-France.xlsx
✅ Saved: Data\processed\ile_de_france_hourly.csv | rows: 26304
Processing: National Consumption.xlsx
✅ Saved: Data\processed\national_consumption_hourly.csv | rows: 26301
Processing: Normandie.xlsx
✅ Saved: Data\processed\normandie_hourly.csv | rows: 26304
Processing: Nouvelle-Aquitaine.xlsx
✅ Saved: Data\processed\nouvelle_aquitaine_hourly.csv | row

In [53]:
def add_timestamp(df: pd.DataFrame, tz: str = "Europe/Paris") -> pd.DataFrame:
    """
    Adds Timestamp column representing the start of the hour.
    Keeps Date/Hour as canonical keys.
    """
    out = df.copy()

    # Ensure types
    out["Hour"] = pd.to_numeric(out["Hour"], errors="coerce").astype("Int64")
    out = out.dropna(subset=["Date", "Hour"])

    # Build "YYYY-MM-DD HH:00:00"
    ts_str = out["Date"].astype(str).str.strip() + " " + out["Hour"].astype(int).astype(str).str.zfill(2) + ":00:00"
    out["Timestamp"] = pd.to_datetime(ts_str, format="%Y-%m-%d %H:%M:%S", errors="coerce")

    # Localize timezone (optional but recommended for France electricity)
    # If you don't want tz-aware datetimes, set tz=None and remove this.
    if tz:
        out["Timestamp"] = out["Timestamp"].dt.tz_localize(tz, ambiguous="infer", nonexistent="shift_forward")

    # Reorder columns nicely
    cols = ["Timestamp", "Date", "Hour", "Perimeter"] + [c for c in out.columns if c not in ["Timestamp","Date","Hour","Perimeter"]]
    return out[cols]


In [54]:
from pathlib import Path

OUT_DIR = Path("Data/processed")

for f in OUT_DIR.glob("*_hourly.csv"):
    df = pd.read_csv(f)
    df = add_timestamp(df, tz=None)
    df.to_csv(f, index=False)
    print("✅ updated:", f.name)


✅ updated: auvergne_rhone_alpes_hourly.csv
✅ updated: bourgogne_franche_comté_hourly.csv
✅ updated: bretagne_hourly.csv
✅ updated: centre_val_de_loire_hourly.csv
✅ updated: grand_est_hourly.csv
✅ updated: hauts_de_france_hourly.csv
✅ updated: ile_de_france_hourly.csv
✅ updated: national_consumption_hourly.csv
✅ updated: normandie_hourly.csv
✅ updated: nouvelle_aquitaine_hourly.csv
✅ updated: occitanie_hourly.csv
✅ updated: paca_hourly.csv
✅ updated: pays_de_la_loire_hourly.csv


In [70]:
sample_out = sorted(OUT_DIR.glob("*_hourly.csv"))[7]
df_check = pd.read_csv(sample_out)
print("Sample:", sample_out.name)
print(df_check.head())
print("Date range:", df_check["Date"].min(), "→", df_check["Date"].max())
print("Hours ok:", df_check["Hour"].between(0, 23).all())
print("Any missing:", df_check.isna().sum().to_dict())


Sample: national_consumption_hourly.csv
             Timestamp        Date  Hour Perimeter  Consumption_MWh
0  2023-01-01 00:00:00  2023-01-01     0    France          46956.5
1  2023-01-01 01:00:00  2023-01-01     1    France          45033.0
2  2023-01-01 02:00:00  2023-01-01     2    France          44923.0
3  2023-01-01 03:00:00  2023-01-01     3    France          42145.0
4  2023-01-01 04:00:00  2023-01-01     4    France          39567.0
Date range: 2023-01-01 → 2025-12-31
Hours ok: True
Any missing: {'Timestamp': 0, 'Date': 0, 'Hour': 0, 'Perimeter': 0, 'Consumption_MWh': 0}


In [40]:
df = pd.read_excel("Data/extracted/National Consumption.xlsx", engine="openpyxl")

print("Columns:", df.columns.tolist())
print("\nDtypes:\n", df.dtypes)

print("\nTail DATE/HOUR raw:")
display(df[["DATE","HOUR"]].tail(30))


Columns: ['DATE', 'HOUR', 'PERIMETER', 'CONSUMPTION']

Dtypes:
 DATE           datetime64[ns]
HOUR                   object
PERIMETER              object
CONSUMPTION            object
dtype: object

Tail DATE/HOUR raw:


,DATE,HOUR
105186,2025-12-31,16:30:00
105187,2025-12-31,16:45:00
105188,2025-12-31,17:00:00
105189,2025-12-31,17:15:00
105190,2025-12-31,17:30:00
105191,2025-12-31,17:45:00
105192,2025-12-31,18:00:00
105193,2025-12-31,18:15:00
105194,2025-12-31,18:30:00
105195,2025-12-31,18:45:00


In [31]:
df = pd.read_excel("Data/extracted/National Consumption.xlsx")

# Build raw datetime string
dt_str = df["DATE"].astype(str) + " " + df["HOUR"].astype(str)

bad = df[pd.to_datetime(dt_str, dayfirst=True, errors="coerce").isna()]

print("Number of bad rows:", len(bad))
bad[["DATE", "HOUR"]].tail(20)


Number of bad rows: 63744


,DATE,HOUR
105196,2025-12-31,19:00:00
105197,2025-12-31,19:15:00
105198,2025-12-31,19:30:00
105199,2025-12-31,19:45:00
105200,2025-12-31,20:00:00
105201,2025-12-31,20:15:00
105202,2025-12-31,20:30:00
105203,2025-12-31,20:45:00
105204,2025-12-31,21:00:00
105205,2025-12-31,21:15:00


In [43]:
# Take only rows after Dec 1 to focus
tail = df.copy()

# show last 200 rows to focus on the problematic zone
tail2 = tail.tail(2000).copy()

display(tail2[["DATE","HOUR"]].head(10))
display(tail2[["DATE","HOUR"]].tail(10))


,DATE,HOUR
103216,2025-12-11,04:00:00
103217,2025-12-11,04:15:00
103218,2025-12-11,04:30:00
103219,2025-12-11,04:45:00
103220,2025-12-11,05:00:00
103221,2025-12-11,05:15:00
103222,2025-12-11,05:30:00
103223,2025-12-11,05:45:00
103224,2025-12-11,06:00:00
103225,2025-12-11,06:15:00


,DATE,HOUR
105206,2025-12-31,21:30:00
105207,2025-12-31,21:45:00
105208,2025-12-31,22:00:00
105209,2025-12-31,22:15:00
105210,2025-12-31,22:30:00
105211,2025-12-31,22:45:00
105212,2025-12-31,23:00:00
105213,2025-12-31,23:15:00
105214,2025-12-31,23:30:00
105215,2025-12-31,23:45:00


In [45]:
dt_try = pd.to_datetime(
    tail2["DATE"].astype(str).str.strip() + " " + tail2["HOUR"].astype(str).str.strip(),
    errors="coerce",
    dayfirst=True
)

bad = tail2[dt_try.isna()][["DATE","HOUR"]].copy()
print("Bad rows in last 2000:", len(bad))
display(bad.tail(30))


Bad rows in last 2000: 1824


,DATE,HOUR
105186,2025-12-31,16:30:00
105187,2025-12-31,16:45:00
105188,2025-12-31,17:00:00
105189,2025-12-31,17:15:00
105190,2025-12-31,17:30:00
105191,2025-12-31,17:45:00
105192,2025-12-31,18:00:00
105193,2025-12-31,18:15:00
105194,2025-12-31,18:30:00
105195,2025-12-31,18:45:00


In [46]:
import datetime as dt

def build_dt_from_date_hour(date_col, hour_col):
    d = date_col
    h = hour_col

    # --- Parse DATE ---
    if np.issubdtype(d.dtype, np.datetime64):
        d_parsed = pd.to_datetime(d).dt.normalize()
    else:
        # string or numeric
        d_parsed = pd.to_datetime(d, errors="coerce", dayfirst=True).dt.normalize()

    # --- Parse HOUR ---
    # Case 1: HOUR is datetime.time objects
    if h.dtype == "object" and len(h) > 0 and isinstance(h.dropna().iloc[0], dt.time):
        h_str = h.apply(lambda x: x.strftime("%H:%M") if isinstance(x, dt.time) else str(x)).astype(str)
        h_str = h_str.str.replace("\u00a0", "", regex=False).str.strip()

    # Case 2: HOUR is numeric (Excel fractional day like 0.5 = 12:00)
    elif np.issubdtype(h.dtype, np.number):
        h_float = pd.to_numeric(h, errors="coerce")
        # Excel time fraction of day -> seconds
        secs = (h_float * 24 * 3600).round()
        h_td = pd.to_timedelta(secs, unit="s")
        return d_parsed + h_td

    # Case 3: HOUR is string like "00:15" / "13:30" / "24:00"
    else:
        h_str = h.astype(str).str.replace("\u00a0", "", regex=False).str.strip().str.lower()
        h_str = h_str.str.replace("h", ":", regex=False)

    # Handle 24:00
    mask_24 = h_str == "24:00"
    h_str = h_str.where(~mask_24, "00:00")

    # Convert hour string to timedelta
    # enforce HH:MM
    h_str = h_str.apply(lambda x: x + ":00" if (x.isdigit() and len(x) <= 2) else x)
    t_parsed = pd.to_datetime(h_str, format="%H:%M", errors="coerce")

    h_td = pd.to_timedelta(t_parsed.dt.hour, unit="h") + pd.to_timedelta(t_parsed.dt.minute, unit="m")
    out = d_parsed + h_td

    # shift 24:00 to next day
    out.loc[mask_24] = out.loc[mask_24] + pd.Timedelta(days=1)
    return out


In [47]:
df["dt"] = build_dt_from_date_hour(df["DATE"], df["HOUR"])

print("NaT count:", df["dt"].isna().sum())
print("Min dt:", df["dt"].min())
print("Max dt:", df["dt"].max())

# show last valid timestamps
display(df.loc[df["dt"].notna(), ["DATE","HOUR","dt"]].tail(20))

# show remaining bad rows (if any)
display(df.loc[df["dt"].isna(), ["DATE","HOUR"]].tail(30))


NaT count: 0
Min dt: 2023-01-01 00:00:00
Max dt: 2025-12-31 23:45:00


,DATE,HOUR,dt
105196,2025-12-31,19:00:00,2025-12-31 19:00:00
105197,2025-12-31,19:15:00,2025-12-31 19:15:00
105198,2025-12-31,19:30:00,2025-12-31 19:30:00
105199,2025-12-31,19:45:00,2025-12-31 19:45:00
105200,2025-12-31,20:00:00,2025-12-31 20:00:00
105201,2025-12-31,20:15:00,2025-12-31 20:15:00
105202,2025-12-31,20:30:00,2025-12-31 20:30:00
105203,2025-12-31,20:45:00,2025-12-31 20:45:00
105204,2025-12-31,21:00:00,2025-12-31 21:00:00
105205,2025-12-31,21:15:00,2025-12-31 21:15:00


,DATE,HOUR


In [48]:
PERIMETER_COL = "PERIMETER"      # may exist; if not, we'll use filename as perimeter
DATE_COL = "DATE"
TIME_COL = "HOUR"
CONS_COL = "CONSUMPTION"

In [49]:
df2 = df.copy()

# perimeter
if PERIMETER_COL:
    df2["Perimeter"] = df2[PERIMETER_COL].astype(str).str.strip()
else:
    df2["Perimeter"] = "France"

# consumption numeric
df2[CONS_COL] = pd.to_numeric(df2[CONS_COL], errors="coerce")
df2 = df2.dropna(subset=["dt", CONS_COL]).sort_values(["Perimeter", "dt"]).reset_index(drop=True)

# delta to next timestamp (hours)
next_dt = df2.groupby("Perimeter")["dt"].shift(-1)
df2["delta_h"] = (next_dt - df2["dt"]).dt.total_seconds() / 3600.0

# Keep only realistic steps (15/30min/1h). If you want, allow up to 1h.
df2["delta_h"] = df2["delta_h"].where(df2["delta_h"] <= 1.0)

# MWh for each interval
df2["energy_mwh"] = df2[CONS_COL] * df2["delta_h"]
df2 = df2.dropna(subset=["energy_mwh"])

# Bucket to hour start
df2["Timestamp"] = df2["dt"].dt.floor("H")
df2["Date"] = df2["Timestamp"].dt.date.astype(str)
df2["Hour"] = df2["Timestamp"].dt.hour.astype(int)

hourly = (df2.groupby(["Date","Hour","Perimeter"], as_index=False)["energy_mwh"]
            .sum()
            .rename(columns={"energy_mwh":"Consumption_MWh"}))

print("Hourly max date:", hourly["Date"].max())
hourly.tail()

Hourly max date: 2025-12-31


C:\Users\sabar\AppData\Local\Temp\ipykernel_6348\388965732.py:25: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df2["Timestamp"] = df2["dt"].dt.floor("H")


,Date,Hour,Perimeter,Consumption_MWh
26296,2025-12-31,19,France,72640.50
26297,2025-12-31,20,France,70071.25
26298,2025-12-31,21,France,67796.00
26299,2025-12-31,22,France,68528.50
26300,2025-12-31,23,France,52420.50


In [83]:
df = pd.read_csv("Data/processed/paca_hourly.csv")

In [85]:
df.loc[df["Perimeter"] == "PACA", "Perimeter"] = "Provence-Alpes-Cote d'Azur"

In [87]:
df.to_csv("Data/processed/paca_hourly.csv", index=False)

# Weather Scraping

In [133]:
REGION_COORDS = {
    "France": (48.8566, 2.3522),  # Paris as national proxy (or use centroid later)

    "Ile-de-France": (48.8566, 2.3522),            # Paris
    "Auvergne-Rhone-Alpes": (45.7640, 4.8357),     # Lyon
    "Bourgogne-Franche-Comte": (47.3220, 5.0415),  # Dijon
    "Bretagne": (48.1173, -1.6778),                # Rennes
    "Centre-Val de Loire": (47.9029, 1.9093),      # Orléans
    "Grand Est": (48.5734, 7.7521),                # Strasbourg
    "Hauts-de-France": (50.6292, 3.0573),          # Lille
    "Normandie": (49.1829, -0.3707),               # Caen
    "Nouvelle-Aquitaine": (44.8378, -0.5792),      # Bordeaux
    "Occitanie": (43.6047, 1.4442),                # Toulouse
    "Pays de la Loire": (47.2184, -1.5536),        # Nantes
    "Provence-Alpes-Cote d'Azur": (43.2965, 5.3698), # Marseille
}

In [134]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry
import time

In [135]:
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

In [137]:
def fetch_openmeteo_hourly_for_regions(
    openmeteo_client,
    region_coords: dict,
    start_date: str,
    end_date: str,
    hourly_vars=None,
    timezone: str = "Europe/Paris",
    sleep_seconds: float = 3
):
    if hourly_vars is None:
        hourly_vars = ["temperature_2m", "wind_speed_10m", "relative_humidity_2m", "shortwave_radiation"]

    url = "https://historical-forecast-api.open-meteo.com/v1/forecast"

    all_parts = []

    for region, (lat, lon) in region_coords.items():
        params = {
            "latitude": lat,
            "longitude": lon,
            "start_date": start_date,
            "end_date": end_date,
            "hourly": hourly_vars,
            "timezone": timezone,
        }

        responses = openmeteo_client.weather_api(url, params=params)
        response = responses[0]

        hourly = response.Hourly()

        # Build hourly timestamp index
        times = pd.date_range(
            start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
            end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
            freq=pd.Timedelta(seconds=hourly.Interval()),
            inclusive="left",
        )

        # Convert to Europe/Paris time (matches your electricity Timestamp)
        times = times.tz_convert(timezone).tz_localize(None).floor("h")

        data = {"Timestamp": times, "Perimeter": region}

        # Extract variables in the exact order requested
        for i, var in enumerate(hourly_vars):
            data[var] = hourly.Variables(i).ValuesAsNumpy()

        df_region = pd.DataFrame(data)
        all_parts.append(df_region)

        print(f"✅ {region}: {len(df_region)} rows")
        time.sleep(sleep_seconds)

    weather = pd.concat(all_parts, ignore_index=True)

    # Optional: ensure hourly unique key per perimeter
    weather = weather.drop_duplicates(subset=["Perimeter", "Timestamp"]).sort_values(["Perimeter", "Timestamp"])

    return weather


In [138]:
weather_df = fetch_openmeteo_hourly_for_regions(
    openmeteo_client=openmeteo,
    region_coords=REGION_COORDS,
    start_date="2023-01-01",
    end_date="2025-12-31",
    hourly_vars=["temperature_2m", "wind_speed_10m", "relative_humidity_2m", "shortwave_radiation"],
    timezone="Europe/Paris",
)

weather_df.to_csv("data/processed/weather_hourly_regions.csv", index=False)
weather_df.head()


✅ France: 26304 rows
✅ Ile-de-France: 26304 rows
✅ Auvergne-Rhone-Alpes: 26304 rows
✅ Bourgogne-Franche-Comte: 26304 rows
✅ Bretagne: 26304 rows
✅ Centre-Val de Loire: 26304 rows
✅ Grand Est: 26304 rows
✅ Hauts-de-France: 26304 rows
✅ Normandie: 26304 rows
✅ Nouvelle-Aquitaine: 26304 rows
✅ Occitanie: 26304 rows
✅ Pays de la Loire: 26304 rows
✅ Provence-Alpes-Cote d'Azur: 26304 rows


,Timestamp,Perimeter,temperature_2m,wind_speed_10m,relative_humidity_2m,shortwave_radiation
52608,2023-01-01 00:00:00,Auvergne-Rhone-Alpes,15.011001,24.627787,68.0,0.0
52609,2023-01-01 01:00:00,Auvergne-Rhone-Alpes,14.861000,24.288662,67.0,0.0
52610,2023-01-01 02:00:00,Auvergne-Rhone-Alpes,14.311001,24.858253,73.0,0.0
52611,2023-01-01 03:00:00,Auvergne-Rhone-Alpes,14.061001,23.732710,75.0,0.0
52612,2023-01-01 04:00:00,Auvergne-Rhone-Alpes,14.061001,22.461807,71.0,0.0


# Merge final Dataset

In [139]:
PROCESSED_DIR = Path("Data/processed")
WEATHER_FILE  = PROCESSED_DIR / "weather_hourly_regions.csv"

OUT_ELEC_MERGED   = PROCESSED_DIR / "all_perimeters_hourly.csv"
OUT_FEATURES_FILE = PROCESSED_DIR / "features_hourly.csv"

In [89]:
elec_files = sorted(PROCESSED_DIR.glob("*_hourly.csv"))
# Exclude weather + any previously merged outputs (if rerunning)
elec_files = [f for f in elec_files if f.name not in {
    "weather_hourly_regions.csv",
    "all_perimeters_hourly.csv",
    "features_hourly.csv"
}]

print("Electricity files found:", len(elec_files))
for f in elec_files:
    print(" -", f.name)

Electricity files found: 13
 - auvergne_rhone_alpes_hourly.csv
 - bourgogne_franche_comté_hourly.csv
 - bretagne_hourly.csv
 - centre_val_de_loire_hourly.csv
 - grand_est_hourly.csv
 - hauts_de_france_hourly.csv
 - ile_de_france_hourly.csv
 - national_hourly.csv
 - normandie_hourly.csv
 - nouvelle_aquitaine_hourly.csv
 - occitanie_hourly.csv
 - paca_hourly.csv
 - pays_de_la_loire_hourly.csv


In [90]:
dfs = []
for f in elec_files:
    df = pd.read_csv(f, parse_dates=["Timestamp"])
    # keep canonical columns (your files already have these)
    keep = [c for c in ["Timestamp", "Perimeter", "Consumption_MWh", "Date", "Hour"] if c in df.columns]
    df = df[keep].copy()

    # clean perimeter
    df["Perimeter"] = df["Perimeter"].astype(str).str.strip()

    # ensure numeric consumption (in case it was read as text)
    df["Consumption_MWh"] = (
        df["Consumption_MWh"].astype(str)
          .str.replace(" ", "", regex=False)
          .str.replace(",", ".", regex=False)
    )
    df["Consumption_MWh"] = pd.to_numeric(df["Consumption_MWh"], errors="coerce")
    df = df.dropna(subset=["Timestamp", "Perimeter", "Consumption_MWh"])

    dfs.append(df)

In [91]:
elec = pd.concat(dfs, ignore_index=True)

# Remove duplicates and sort
elec = elec.drop_duplicates(subset=["Perimeter", "Timestamp"]).sort_values(["Perimeter", "Timestamp"]).reset_index(drop=True)

# Save merged electricity
elec.to_csv(OUT_ELEC_MERGED, index=False)
print("✅ Saved merged electricity:", OUT_ELEC_MERGED, "| rows:", len(elec))

✅ Saved merged electricity: Data\processed\all_perimeters_hourly.csv | rows: 341949


In [140]:
weather = pd.read_csv(WEATHER_FILE, parse_dates=["Timestamp"])
weather["Perimeter"] = weather["Perimeter"].astype(str).str.strip()

In [124]:
print(elec["Perimeter"].unique())

['Auvergne-Rhône-Alpes' 'Bourgogne-Franche-Comté' 'Bretagne'
 'Centre-Val-de-Loire' 'France' 'Grand-Est' 'Hauts-de-France'
 'Ile-de-France' 'Normandie' 'Nouvelle-Aquitaine' 'Occitanie'
 'Pays-de-la-Loire' "Provence-Alpes-Cote d'Azur"]


In [125]:
print(weather["Perimeter"].unique())

['Auvergne-Rhone-Alpes' 'Bourgogne-Franche-Comte' 'Bretagne'
 'Centre-Val de Loire' 'France' 'Grand Est' 'Hauts-de-France'
 'Ile-de-France' 'Normandie' 'Nouvelle-Aquitaine' 'Occitanie'
 'Pays de la Loire' "Provence-Alpes-Cote d'Azur"]


In [141]:
RENAME = {"Auvergne-Rhone-Alpes": "Auvergne-Rhône-Alpes", "Bourgogne-Franche-Comte": "Bourgogne-Franche-Comté",
          "Centre-Val de Loire" : "Centre-Val-de-Loire", "Grand Est": "Grand-Est", "Pays de la Loire":"Pays-de-la-Loire"}  # example
elec["Perimeter"] = elec["Perimeter"].replace(RENAME)
weather["Perimeter"] = weather["Perimeter"].replace(RENAME)

In [142]:
missing_weather = sorted(set(elec["Perimeter"].unique()) - set(weather["Perimeter"].unique()))
if missing_weather:
    print("⚠️ Perimeters missing in weather:", missing_weather)


In [143]:
weather.dtypes

Timestamp               datetime64[ns]
Perimeter                       object
temperature_2m                 float64
wind_speed_10m                 float64
relative_humidity_2m           float64
shortwave_radiation            float64
dtype: object

In [112]:
print("Electricity minute distribution:")
print(elec["Timestamp"].dt.minute.value_counts().head())

print("Weather minute distribution:")
print(weather["Timestamp"].dt.minute.value_counts().head())


Electricity minute distribution:
Timestamp
0    341949
Name: count, dtype: int64
Weather minute distribution:
Timestamp
0.0    143208
Name: count, dtype: int64


In [ ]:
# elec["Timestamp"] = pd.to_datetime(elec["Timestamp"]).dt.tz_localize(None)
# weather["Timestamp"] = pd.to_datetime(weather["Timestamp"]).dt.tz_convert("Europe/Paris").dt.tz_localize(None)

In [144]:
merged = elec.merge(weather, on=["Timestamp", "Perimeter"], how="left")

In [145]:
if "temperature_2m" in merged.columns:
    print("Weather missing rate (temperature_2m):", f"{merged['temperature_2m'].isna().mean():.2%}")

Weather missing rate (temperature_2m): 0.00%


In [146]:
merged.to_csv(OUT_FEATURES_FILE, index=False)
print("✅ Saved merged features:", OUT_FEATURES_FILE, "| rows:", len(merged))

✅ Saved merged features: Data\processed\features_hourly.csv | rows: 341910


In [147]:
import pandas as pd

df = pd.read_csv("Data/processed/features_hourly.csv", parse_dates=["Timestamp"])

print("Rows:", len(df))
print("Perimeters:", df["Perimeter"].nunique())
print("Date range:", df["Timestamp"].min(), "→", df["Timestamp"].max())

Rows: 341910
Perimeters: 13
Date range: 2023-01-01 00:00:00 → 2025-12-31 23:00:00


In [148]:
dups = df.duplicated(subset=["Perimeter", "Timestamp"]).sum()
print("Duplicate (Perimeter, Timestamp):", dups)

Duplicate (Perimeter, Timestamp): 39


In [149]:
df = pd.read_csv("Data/processed/features_hourly.csv", parse_dates=["Timestamp"])

weather_cols = ["temperature_2m", "wind_speed_10m", "relative_humidity_2m", "shortwave_radiation"]

df[weather_cols].describe()


,temperature_2m,wind_speed_10m,relative_humidity_2m,shortwave_radiation
count,341910.000000,341910.000000,341910.000000,341910.000000
mean,13.875372,10.075074,73.168251,144.936181
std,7.270721,6.187411,17.353316,222.191098
min,-7.267500,0.000000,0.000000,0.000000
25%,8.900000,5.447788,62.000000,0.000000
50%,13.500000,9.007196,77.000000,6.000000
75%,18.793499,13.378250,87.000000,228.250000
max,41.245502,55.968136,100.000000,964.250000
